# Optimizer comparison on Breast Cancer dataset with SVM model

In [1]:
import torch
import torch.nn as nn

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import SVM
from src.utils import build_sampler, data_load_and_prep
from src.training import train
from src.newton_optimizer import NewtonOptimizer
from src.losses import SVMSquaredHingeLoss

EXPERIMENT_NAME = "optimizer-comparison-svm-bc"

In [2]:
train_loader, test_loader = data_load_and_prep("breast_cancer", test_size=0.3, random_state=42, batch_size='full')

## Adam

In [3]:
loss_fn = SVMSquaredHingeLoss(C=1.0) 
adam_model = SVM(input_dim=30, output_dim=1)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Adam optimizaer

In [4]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=30, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/13 10:30:52 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 000 | train_loss=166.3206 | train_acc=0.671 | test_loss=82.3072 | test_acc=0.737 | 
Epoch 005 | train_loss=99.1680 | train_acc=0.912 | test_loss=78.9323 | test_acc=0.924 | 
Epoch 010 | train_loss=79.1356 | train_acc=0.950 | test_loss=57.7690 | test_acc=0.924 | 
Epoch 015 | train_loss=62.0764 | train_acc=0.957 | test_loss=37.9724 | test_acc=0.918 | 
Epoch 020 | train_loss=61.2734 | train_acc=0.960 | test_loss=34.3991 | test_acc=0.930 | 
Epoch 025 | train_loss=53.8929 | train_acc=0.967 | test_loss=32.2011 | test_acc=0.936 | 
Epoch 029 | train_loss=52.7128 | train_acc=0.972 | test_loss=32.9661 | test_acc=0.947 | 


{'run_id': 'c4d87e9e5fc84a4cad861dcc2af046ff',
 'training_time_sec': 0.3967885199999728,
 'final_train_loss': 52.71281051635742,
 'final_test_loss': 32.9660758972168,
 'final_train_metric': 0.9723618090452262,
 'final_test_metric': 0.9473684210526315,
 'optimizer': 'Adam',
 'seed': None}

## l-BFGS optimizer

In [5]:
loss_fn = SVMSquaredHingeLoss(C=1.0)
lbfgs_model = SVM(input_dim=30, output_dim=1)
classical_device = "cuda" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.5,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [6]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

Epoch 000 | train_loss=134.6286 | train_acc=0.791 | test_loss=57.4668 | test_acc=0.749 | 
Epoch 005 | train_loss=56.8129 | train_acc=0.972 | test_loss=31.4805 | test_acc=0.953 | 
Epoch 010 | train_loss=45.3516 | train_acc=0.957 | test_loss=25.2270 | test_acc=0.965 | 
Epoch 015 | train_loss=42.4757 | train_acc=0.965 | test_loss=23.4498 | test_acc=0.965 | 
Epoch 020 | train_loss=41.1732 | train_acc=0.972 | test_loss=23.1244 | test_acc=0.947 | 
Epoch 025 | train_loss=40.5415 | train_acc=0.975 | test_loss=23.2390 | test_acc=0.942 | 
Epoch 029 | train_loss=40.3887 | train_acc=0.975 | test_loss=23.1314 | test_acc=0.942 | 


2026/04/13 10:30:57 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


{'run_id': '4442c1bb95d347aaa6c26bb3661a343f',
 'training_time_sec': 0.2372202289998313,
 'final_train_loss': 40.38871765136719,
 'final_test_loss': 23.131366729736328,
 'final_train_metric': 0.9748743718592965,
 'final_test_metric': 0.9415204678362573,
 'optimizer': 'LBFGS',
 'seed': None}

## Newton-Raphson optimizer

In [7]:
loss_fn = SVMSquaredHingeLoss(C=1.0)
newton_model = SVM(input_dim=30, output_dim=1)
classical_device = "cuda" 
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                              lr=0.1,
                              max_iter=1,
                              damping=1e-4,
                              )

In [8]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

Epoch 000 | train_loss=196.0063 | train_acc=0.553 | test_loss=93.3328 | test_acc=0.515 | 
Epoch 005 | train_loss=94.4806 | train_acc=0.955 | test_loss=46.2165 | test_acc=0.918 | 
Epoch 010 | train_loss=59.0808 | train_acc=0.975 | test_loss=30.5594 | test_acc=0.965 | 
Epoch 015 | train_loss=46.7377 | train_acc=0.975 | test_loss=25.5555 | test_acc=0.959 | 
Epoch 020 | train_loss=42.4339 | train_acc=0.972 | test_loss=24.0798 | test_acc=0.947 | 
Epoch 025 | train_loss=40.9332 | train_acc=0.975 | test_loss=23.7240 | test_acc=0.947 | 


2026/04/13 10:31:03 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=40.4757 | train_acc=0.975 | test_loss=23.6897 | test_acc=0.947 | 


{'run_id': 'e68da347daa5467ea640095d9cfee7db',
 'training_time_sec': 0.9967930110001362,
 'final_train_loss': 40.475704193115234,
 'final_test_loss': 23.689655303955078,
 'final_train_metric': 0.9748743718592965,
 'final_test_metric': 0.9473684210526315,
 'optimizer': 'NewtonOptimizer',
 'seed': None}

## Quadratic Annealing optimizer (cpu)

In [9]:
loss_fn = SVMSquaredHingeLoss(C=1.0)
model = SVM(input_dim=30, output_dim=1)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [10]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-optimizer"
)

Epoch 000 | train_loss=113.2646 | train_acc=0.827 | test_loss=52.0112 | test_acc=0.760 | 
Epoch 005 | train_loss=52.5507 | train_acc=0.950 | test_loss=28.3659 | test_acc=0.947 | 
Epoch 010 | train_loss=46.8344 | train_acc=0.962 | test_loss=27.1170 | test_acc=0.942 | 
Epoch 015 | train_loss=44.8029 | train_acc=0.965 | test_loss=23.9107 | test_acc=0.942 | 
Epoch 020 | train_loss=44.6731 | train_acc=0.962 | test_loss=25.0709 | test_acc=0.936 | 
Epoch 025 | train_loss=44.6731 | train_acc=0.962 | test_loss=25.0709 | test_acc=0.936 | 
Epoch 029 | train_loss=44.6731 | train_acc=0.962 | test_loss=25.0709 | test_acc=0.936 | 


2026/04/13 10:31:10 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


{'run_id': 'cde50d6067754e698111eaf38253dba9',
 'training_time_sec': 0.658234345999972,
 'final_train_loss': 44.67307662963867,
 'final_test_loss': 25.07086753845215,
 'final_train_metric': 0.9623115577889447,
 'final_test_metric': 0.935672514619883,
 'optimizer': 'QuadraticAnnealingOptimizer',
 'seed': None}

## Quadratic Annealing optimizer (cpu + momentum)

In [11]:
loss_fn = SVMSquaredHingeLoss(C=1.0)
model = SVM(input_dim=30, output_dim=1)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
    beta=0.9,
)

In [12]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-momentum-optimizer"
)

Epoch 000 | train_loss=92.0897 | train_acc=0.899 | test_loss=40.8011 | test_acc=0.883 | 
Epoch 005 | train_loss=92.0897 | train_acc=0.899 | test_loss=40.8011 | test_acc=0.883 | 
Epoch 010 | train_loss=92.0897 | train_acc=0.899 | test_loss=40.8011 | test_acc=0.883 | 
Epoch 015 | train_loss=92.0897 | train_acc=0.899 | test_loss=40.8011 | test_acc=0.883 | 


2026/04/13 10:31:14 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 020 | train_loss=92.0897 | train_acc=0.899 | test_loss=40.8011 | test_acc=0.883 | 
Epoch 025 | train_loss=92.0897 | train_acc=0.899 | test_loss=40.8011 | test_acc=0.883 | 
Epoch 029 | train_loss=92.0897 | train_acc=0.899 | test_loss=40.8011 | test_acc=0.883 | 


{'run_id': '0fcbf75623c9498dbf3761753daf32b6',
 'training_time_sec': 0.7530516679998982,
 'final_train_loss': 92.08972930908203,
 'final_test_loss': 40.80107498168945,
 'final_train_metric': 0.8994974874371859,
 'final_test_metric': 0.8830409356725146,
 'optimizer': 'QuadraticAnnealingOptimizer',
 'seed': None}